# Rumney Marsh — stigmergic routing (Monte Carlo)

Spatial Diffusion $D$ and Canalization Ratio $C$ per phenotype, mean ± 1σ across seeds at every simulation step, in the book-chapter style.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------------------------------------------------------
# Locate project root
# -----------------------------------------------------------------------------
cwd = Path.cwd().resolve()

for parent in [cwd, *cwd.parents]:
    if (parent / "beaversim").is_dir():
        project_root = parent
        break
else:
    raise RuntimeError("Could not locate project root containing 'beaversim'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

from beaversim.ral.backend.beavers_visualizer_backend import BeaversVisualizerBackend

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. CORE EXPERIMENT CONFIGURATION
# ---------------------------------------------------------
backend_file = "final_backend.pkl.gz"
hours_per_year = 365 * 24

linestyles = [
    "solid",
    (0, (8, 3)),
    (0, (10, 3, 2, 3)),
    (0, (1.2, 2.2)),
]

colors = {
    "explorer": "#1f77b4",
    "builder": "#d62728",
}

# ---------------------------------------------------------
# 2. GLOBAL MATPLOTLIB STYLING (rcParams)
# ---------------------------------------------------------
plt.rcParams.update({
    # --- FONT SIZES ---
    "font.size": 12,               # Base font size for everything
    "axes.titlesize": 16,          # Font size for ax.set_title()
    "axes.labelsize": 14,          # Font size for x and y axis labels
    "xtick.labelsize": 12,         # Font size for X-axis tick marks
    "ytick.labelsize": 12,         # Font size for Y-axis tick marks
    "legend.fontsize": 12,         # Font size for legend text
    "legend.title_fontsize": 13,   # Font size for legend title
    
    # --- LINE WIDTHS & SIZES ---
    "lines.linewidth": 2.5,        # Thickness of your plotted lines
    "lines.markersize": 6,         # Size of markers (if you use scatter/plot markers)
    
    # --- PLOT BOX (SPINES) ---
    "axes.linewidth": 1.5,         # Thickness of the box surrounding the plot
    "axes.edgecolor": "black",     # Color of the box
    
    # --- GRID STYLING ---
    "axes.grid": True,             # Turn the grid on by default
    "grid.linewidth": 1.0,         # Thickness of the grid lines
    "grid.alpha": 0.5,             # Transparency of the grid (0 to 1)
    "grid.color": "#b0b0b0",       # A nice soft grey for the grid
    "grid.linestyle": "--",        # Dashed grid lines
    
    # --- LEGEND STYLING ---
    "legend.frameon": True,        # Draw a box around the legend
    "legend.edgecolor": "black",   # Color of the legend box
    "legend.fancybox": False,      # Sharp corners for the legend (True makes them rounded)
    
    # --- RESOLUTION ---
    "figure.dpi": 150,             # Higher resolution for inline notebook rendering
    "savefig.dpi": 300,            # High resolution for saving PDFs/PNGs for your paper
})

In [ ]:
# Every subfolder of marsh_root with a montecarlo/ directory is one condition
marsh_root = project_root / "output" / "simulations" / "rumney_marsh_springer"
base_dirs = sorted(d for d in marsh_root.iterdir() if (d / "montecarlo").is_dir())

hours_per_month = hours_per_year / 12  # 730 h

In [ ]:
sim_results = {}

def seed_id(seed_dir):
    return int(seed_dir.name.split("_")[-1])

for base_dir in base_dirs:
    seed_dirs = sorted([d for d in (base_dir / "montecarlo").iterdir() if d.name.startswith("seed_")], key=seed_id)
    seeds = []

    for seed_dir in seed_dirs:
        backend_path = seed_dir / backend_file
        if not backend_path.exists():
            continue
        try:
            back = BeaversVisualizerBackend.load_backend_pickle(str(backend_path))
            seeds.append({
                "seed": seed_id(seed_dir),
                "timedelta": float(getattr(back, "_timedelta", 1.0)),
                "plotting": getattr(back._environment, "_plotting_stats", {}) or {},
            })
        except Exception as e:
            print("Error:", seed_dir.name, e)

    if seeds:
        sim_results[base_dir.name] = seeds
        print("Loaded:", base_dir.name, "seeds", [s["seed"] for s in seeds])

In [ ]:
# ---------------------------------------------------------
# DESIGN STYLE (book chapter)
# ---------------------------------------------------------
INK, MID = "#111111", "#7a7a7a"

plt.rcParams.update({
    "font.family": "Inter",
    "font.weight": "light",
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "axes.edgecolor": INK,
    "axes.labelcolor": MID,
    "axes.labelsize": 8,
    "axes.grid": False,
    "xtick.color": MID, "ytick.color": MID,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.direction": "out", "ytick.direction": "out",
    "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "lines.dash_capstyle": "round", "lines.dash_joinstyle": "round",
    "hatch.linewidth": 0.3,
    "savefig.dpi": 300,
})

# One entry per condition, in folder order: line colour, band pattern colour, line style, band pattern
condition_styles = [
    dict(hue="#24733f", pattern_color="#b3d3bb", ls="solid",          hatch="////////////"),  # forest green, hatch
    dict(hue="#c99a3c", pattern_color="#e8cd94", ls=(0, (0.1, 2.2)), hatch="............"),  # ochre, stipple
    dict(hue="#555555", pattern_color="#cccccc", ls=(0, (4, 2)),      hatch="xxxxxx"),        # grey fallback
]


def spaced(s):
    """Letter-spaced caps for headings."""
    return "  ".join(" ".join(w) for w in s.upper().split())


def condition_label(cond):
    """'10agents_6months_vegetation_quality' -> 'quality'."""
    return cond.split("vegetation_")[-1]


def smooth(y, window):
    """Centered moving average over `window` samples (edges padded)."""
    if window <= 1:
        return y
    ypad = np.pad(y, (window // 2, window - 1 - window // 2), mode="edge")
    return np.convolve(ypad, np.ones(window) / window, mode="valid")


def plot_design_panel(ax, key, smooth_h=1, skip_h=0):
    """Mean line + patterned ±1σ band per condition, direct labels at the line ends."""
    labels = []
    for i, (cond, seeds) in enumerate(sim_results.items()):
        st = condition_styles[min(i, len(condition_styles) - 1)]
        dt = seeds[0]["timedelta"]
        n = min(len(s["plotting"][key]) for s in seeds)
        Y = np.vstack([np.asarray(s["plotting"][key][:n], float) for s in seeds])
        w = max(1, int(round(smooth_h / dt)))
        mu, sd = smooth(Y.mean(axis=0), w), smooth(Y.std(axis=0), w)
        x = np.arange(n) * dt / hours_per_month
        keep = x >= skip_h / hours_per_month
        x, mu, sd = x[keep], mu[keep], sd[keep]

        ax.fill_between(x, np.maximum(0, mu - sd), mu + sd, facecolor="none",
                        edgecolor=st["pattern_color"], hatch=st["hatch"], linewidth=0)
        ax.plot(x, mu, color=st["hue"], lw=1.8, ls=st["ls"])
        labels.append([mu[-1], f"{condition_label(cond)}  n={len(seeds)}", st["hue"]])

    # Direct labels at the line ends, pushed apart so they never overlap
    lo, hi = ax.get_ylim()
    gap = 0.07 * (hi - lo)
    labels.sort()
    for a, b in zip(labels, labels[1:]):
        if b[0] - a[0] < gap:
            mid = (a[0] + b[0]) / 2
            a[0], b[0] = mid - gap / 2, mid + gap / 2
    for y, txt, col in labels:
        ax.annotate(txt, (x[-1], y), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=7.5, color=col, annotation_clip=False)

    ax.set_xlim(0, x[-1])
    ax.set_xticks(range(0, int(round(x[-1])) + 1))
    ax.set_xlabel("months")

In [ ]:
# ---------------------------------------------------------
# Two figures (D, then C), columns = Explorer / Builder
# ---------------------------------------------------------
SAVE_FIG = False

# C is smoothed with a 1-week moving average and its first day (C jumps 0 <-> 1 while
# almost no cells carry traffic) is hidden. Mention both in the caption.
figures = [
    dict(metric="spatial_footprint",  title="spatial diffusion",  ylabel="cells visited",
         smooth_h=1,   skip_h=0,  fname="stigmergy_D.pdf"),
    dict(metric="canalization_ratio", title="canalization ratio", ylabel="share of traffic on top 5% cells",
         smooth_h=168, skip_h=24, fname="stigmergy_C.pdf"),
]

for f in figures:
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.0))
    for ax, p in zip(axes, ["explorer", "builder"]):
        plot_design_panel(ax, f"{p}_{f['metric']}", smooth_h=f["smooth_h"], skip_h=f["skip_h"])
        ax.set_title(spaced(p), loc="left", fontsize=7.5, color=MID, pad=10)
    axes[0].set_ylabel(f["ylabel"])
    fig.text(0.01, 0.98, spaced(f["title"]), fontsize=8.5, color=INK, weight="medium", va="top")
    fig.tight_layout(rect=(0, 0, 0.96, 0.92), w_pad=4)

    if SAVE_FIG:
        fig.savefig(marsh_root / f["fname"], bbox_inches="tight")
    plt.show()